# Hybrid Movie Recommendation - Notebook sạch theo đúng quy trình

Notebook này được sắp xếp lại theo đúng thứ tự bạn yêu cầu:

1. Import thư viện  
2. Đọc dữ liệu  
3. Kiểm tra dữ liệu  
4. Tiền xử lý dữ liệu  
5. Trích chọn đặc trưng TF-IDF  
6. Chia train/dev/test  
7. Huấn luyện SVD  
8. Đánh giá SVD bằng RMSE/MAE  
9. Xây dựng Hybrid Recommendation  
10. Đánh giá Top-K  
11. Demo gợi ý  
12. Lưu kết quả và model  

Mục tiêu của bản này là **không để code bị lẫn lộn**, **không viết lặp nhiều hàm giống nhau**, và mỗi phần đều có ghi chú giải thích trước khi chạy.

## 1. Import thư viện

Phần này chỉ dùng để nạp các thư viện cần thiết cho toàn bộ notebook.

- `pandas`, `numpy`: xử lý dữ liệu dạng bảng và tính toán số học.
- `train_test_split`: chia dữ liệu thành train/dev/test.
- `TfidfVectorizer`: biến nội dung phim dạng chữ thành vector số.
- `cosine_similarity`: tính độ giống nhau giữa hồ sơ sở thích user và phim.
- `mean_squared_error`, `mean_absolute_error`: đánh giá lỗi dự đoán rating của SVD.
- `vstack`: ghép các vector TF-IDF của nhiều phim.
- `surprise`: thư viện dùng để huấn luyện SVD cho hệ gợi ý.

In [2]:
import os
import pickle

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error

from scipy.sparse import vstack

from surprise import Dataset, Reader, SVD


In [3]:
Nếu máy chưa cài `scikit-surprise`, chạy cell dưới một lần. Nếu đã cài rồi thì bỏ qua.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
# !pip install scikit-surprise


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
## 2. Đọc dữ liệu

Ở phần này, notebook đọc 4 file dữ liệu gốc từ thư mục `data`:

- `ratings.csv`: dữ liệu đánh giá phim của người dùng.
- `movies.csv`: thông tin phim gồm `movieId`, `title`, `genres`.
- `tags.csv`: tag do người dùng gắn cho phim.
- `links.csv`: liên kết tới các nguồn dữ liệu khác.

Trong mô hình chính, nhóm dùng `ratings.csv`, `movies.csv`, `tags.csv`. File `links.csv` được đọc để kiểm tra đầy đủ dữ liệu nhưng không dùng trực tiếp trong mô hình Hybrid.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   userId     3683 non-null   int64 
 1   movieId    3683 non-null   int64 
 2   tag        3683 non-null   object
 3   timestamp  3683 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 115.2+ KB


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [6]:
ratings_df = pd.read_csv("data/ratings.csv")
movies_df = pd.read_csv("data/movies.csv")
tags_df = pd.read_csv("data/tags.csv")
links_df = pd.read_csv("data/links.csv")

print("Ratings:", ratings_df.shape)
print("Movies:", movies_df.shape)
print("Tags:", tags_df.shape)
print("Links:", links_df.shape)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


Kiểm tra nhanh vài dòng đầu của từng bảng dữ liệu.

In [7]:
ratings_df.head()


In [ ]:
movies_df.head()


In [ ]:
tags_df.head()


## 3. Kiểm tra dữ liệu

Phần này kiểm tra dữ liệu trước khi tiền xử lý:

1. Kiểm tra dữ liệu trùng lặp.
2. Chuẩn bị bảng rating cho SVD.
3. Kiểm tra số lượng user, số lượng phim, số lượt rating.
4. Tính độ thưa của ma trận rating.

Độ thưa cao nghĩa là đa số user chưa rating đa số phim. Đây là lý do cần dùng SVD/Funk SVD để học trên các rating đã có.

In [ ]:
print("Ratings duplicate:", ratings_df.duplicated().sum())
print("Movies duplicate:", movies_df.duplicated().sum())
print("Tags duplicate:", tags_df.duplicated().sum())


Chuẩn bị dữ liệu rating cho SVD. Mô hình SVD chỉ cần 3 cột:

- `userId`
- `movieId`
- `rating`

In [8]:
# Xử lý DL cho SVD
# Đọc dữ liệu ratings, dùng copy không pandas sẽ sinh ra cái gì đó? 
ratings_svd = ratings_df[
    [
        "userId",
        "movieId",
        "rating"
    ]
].copy()
ratings_svd.info()
ratings_svd.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   userId   100836 non-null  int64  
 1   movieId  100836 non-null  int64  
 2   rating   100836 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 2.3 MB


,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


In [9]:
# kiểm tra xem có bị ngoài khoảng (0.5 - 5)?
ratings_svd["rating"].describe()

count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64

In [10]:
# kiểm tra số lượng User
ratings_svd["userId"].nunique()

610

In [11]:
ratings_svd["movieId"].nunique()

9724

In [12]:
ratings_svd = ratings_df[
    ["userId", "movieId", "rating"]
].copy()

ratings_svd.info()
ratings_svd.head()


In [13]:
ratings_svd["rating"].describe()


,movieId,title,genres,year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0
4,5,Father of the Bride Part II (1995),Comedy,1995.0


In [14]:
# Xóa năm khỏi title
movies_df["title"] = movies_df["title"].str.replace(
    r"\(\d{4}\)",
    "",
    regex=True
).str.strip()
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,2,Jumanji,Adventure|Children|Fantasy,1995.0
2,3,Grumpier Old Men,Comedy|Romance,1995.0
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995.0
4,5,Father of the Bride Part II,Comedy,1995.0


In [15]:
# Tính sparsity để xem độ thưa là bao nhiêu, và nêu lý do chọn SVD và SVD sẽ xử lý tốt dữ liệu thưa (không bắt buộc, chỉ tính để biết nó thưa và SVD làm tốt với DL thưa)
n_users = ratings_svd["userId"].nunique()
n_movies = ratings_svd["movieId"].nunique()
n_ratings = len(ratings_svd)

print("Số user:", n_users)
print("Số phim có rating:", n_movies)
print("Số lượt rating:", n_ratings)

sparsity = (
    1 - n_ratings / (n_users * n_movies)
) * 100

print(f"Sparsity: {sparsity:.2f}%")


Sparsity: 98.30%


In [16]:
## 4. Tiền xử lý dữ liệu

Phần này xử lý dữ liệu phim và tag để tạo cột `content`.

Cột `content` sẽ là văn bản tổng hợp từ:

```text
title + genres + tag
```

Sau đó cột này được dùng cho TF-IDF ở phần tiếp theo.

array(['Adventure|Animation|Children|Comedy|Fantasy',
       'Adventure|Children|Fantasy', 'Comedy|Romance',
       'Comedy|Drama|Romance', 'Comedy', 'Action|Crime|Thriller',
       'Adventure|Children', 'Action', 'Action|Adventure|Thriller',
       'Comedy|Horror', 'Adventure|Animation|Children', 'Drama',
       'Action|Adventure|Romance', 'Crime|Drama', 'Drama|Romance',
       'Action|Comedy|Crime|Drama|Thriller', 'Comedy|Crime|Thriller',
       'Crime|Drama|Horror|Mystery|Thriller', 'Drama|Sci-Fi',
       'Children|Drama', 'Adventure|Drama|Fantasy|Mystery|Sci-Fi',
       'Mystery|Sci-Fi|Thriller', 'Children|Comedy', 'Drama|War',
       'Action|Crime|Drama', 'Action|Adventure|Fantasy',
       'Comedy|Drama|Thriller', 'Mystery|Thriller',
       'Animation|Children|Drama|Musical|Romance',
       'Crime|Mystery|Thriller', 'Adventure|Drama', 'Drama|Thriller',
       'Comedy|Crime', 'Action|Sci-Fi|Thriller',
       'Action|Comedy|Horror|Thriller', 'Comedy|Drama', 'Documentary',
       'Ac

In [17]:
### 4.1. Tách năm phát hành từ title

Trong `movies.csv`, title thường có dạng:

```text
Toy Story (1995)
```

Ta tách `1995` ra cột `year` để title sạch hơn.

In [18]:
movies_df["year"] = movies_df["title"].str.extract(r"\((\d{4})\)")

movies_df["year"] = pd.to_numeric(
    movies_df["year"],
    errors="coerce"
)

movies_df.head()


### 4.2. Xóa năm khỏi title

Sau khi đã tách năm, ta xóa phần `(1995)` khỏi title để nội dung phim gọn hơn.

In [ ]:
movies_df["title"] = (
    movies_df["title"]
    .str.replace(r"\(\d{4}\)", "", regex=True)
    .str.strip()
)

movies_df.head()


### 4.3. Chuẩn hóa genres

Một số phim có giá trị `(no genres listed)`, nghĩa là không có thể loại. Ta đổi giá trị này thành chuỗi rỗng.

Dữ liệu gốc dùng dấu `|` để ngăn cách thể loại, ví dụ:

```text
Adventure|Animation|Children
```

Ta đổi dấu `|` thành khoảng trắng để đưa vào TF-IDF.

In [ ]:
count_no_genres = len(
    movies_df[movies_df["genres"] == "(no genres listed)"]
)

print("Số phim không có genres:", count_no_genres)

movies_df["genres"] = movies_df["genres"].replace(
    "(no genres listed)",
    ""
)

In [19]:
# Kiểm tra lại đã thay chưa
count = len(
    movies_df[movies_df["genres"] == "(no genres listed)"]
)
print(count)

0


In [20]:
# Thay dấu | trong genres bằng khoảng trắng
movies_df["genres"] = movies_df["genres"].str.replace(
    "|",
    " ",
    regex=False
)

movies_df.head()


,movieId,title,genres,year
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,1995.0
1,2,Jumanji,Adventure Children Fantasy,1995.0
2,3,Grumpier Old Men,Comedy Romance,1995.0
3,4,Waiting to Exhale,Comedy Drama Romance,1995.0
4,5,Father of the Bride Part II,Comedy,1995.0


### 4.4. Chuẩn hóa tag

Tag là các từ khóa do user gắn cho phim. Ta đưa tag về chữ thường và xóa khoảng trắng thừa.

In [21]:
# Xử lý tags
# Chuẩn hóa thành chữ thường 
tags_df["tag"] = tags_df["tag"].str.lower()
tags_df.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,boxing story,1445715207
4,2,89774,mma,1445715200


In [22]:
# Loại bỏ khoảng trắng thừa, ở 2 đầu nếu người dùng lỡ bấm space trước/sau khi viết 
tags_df["tag"] = tags_df["tag"].str.strip()
tags_df.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,boxing story,1445715207
4,2,89774,mma,1445715200


In [23]:
movie_tags["tag"] = movie_tags["tag"].apply(
    lambda x: " ".join(
        dict.fromkeys(x.split())
    )
)

movie_tags.head()


,movieId,tag
0,1,pixar pixar fun
1,2,fantasy magic board game robin williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake


### 4.7. Ghép tag vào dữ liệu phim

Ta ghép bảng `movie_tags` vào `movies_df` để mỗi phim có thêm cột `tag`.

In [24]:
movies_nlp = movies_df.merge(
    movie_tags,
    on="movieId",
    how="left"
)

movies_nlp["tag"] = movies_nlp["tag"].fillna("")

movies_nlp.head()


,movieId,title,genres,year,tag
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,1995.0,pixar pixar fun
1,2,Jumanji,Adventure Children Fantasy,1995.0,fantasy magic board game robin williams game
2,3,Grumpier Old Men,Comedy Romance,1995.0,moldy old
3,4,Waiting to Exhale,Comedy Drama Romance,1995.0,NaN
4,5,Father of the Bride Part II,Comedy,1995.0,pregnancy remake


### 4.8. Tạo cột content

Cột `content` là dữ liệu văn bản chính dùng cho TF-IDF.

Nó gồm:

```text
title + genres + tag
```

In [25]:
# Xử lý Null
movies_nlp["tag"] = (
    movies_nlp["tag"]
    .fillna("")
)
movies_nlp.head()

,movieId,title,genres,year,tag
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,1995.0,pixar pixar fun
1,2,Jumanji,Adventure Children Fantasy,1995.0,fantasy magic board game robin williams game
2,3,Grumpier Old Men,Comedy Romance,1995.0,moldy old
3,4,Waiting to Exhale,Comedy Drama Romance,1995.0,
4,5,Father of the Bride Part II,Comedy,1995.0,pregnancy remake


In [26]:
# Ghép title, genres, tag thành 1 cột content 
movies_nlp["content"] = (
    movies_nlp["title"]
    + " "
    + movies_nlp["genres"]
    + " "
    + movies_nlp["tag"]
)

movies_nlp.head()


,movieId,title,genres,year,tag,content
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,1995.0,pixar pixar fun,Toy Story Adventure Animation Children Comedy ...
1,2,Jumanji,Adventure Children Fantasy,1995.0,fantasy magic board game robin williams game,Jumanji Adventure Children Fantasy fantasy mag...
2,3,Grumpier Old Men,Comedy Romance,1995.0,moldy old,Grumpier Old Men Comedy Romance moldy old
3,4,Waiting to Exhale,Comedy Drama Romance,1995.0,,Waiting to Exhale Comedy Drama Romance
4,5,Father of the Bride Part II,Comedy,1995.0,pregnancy remake,Father of the Bride Part II Comedy pregnancy r...


### 4.9. Làm sạch content

Các bước làm sạch gồm:

1. Chuyển về chữ thường.
2. Kiểm tra ký tự đặc biệt.
3. Loại bỏ ký tự đặc biệt.
4. Chuẩn hóa khoảng trắng.

In [27]:
movies_nlp["content"] = (
    movies_nlp["content"]
    .str.lower()
)
movies_nlp.head()

,movieId,title,genres,year,tag,content
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,1995.0,pixar pixar fun,toy story adventure animation children comedy ...
1,2,Jumanji,Adventure Children Fantasy,1995.0,fantasy magic board game robin williams game,jumanji adventure children fantasy fantasy mag...
2,3,Grumpier Old Men,Comedy Romance,1995.0,moldy old,grumpier old men comedy romance moldy old
3,4,Waiting to Exhale,Comedy Drama Romance,1995.0,,waiting to exhale comedy drama romance
4,5,Father of the Bride Part II,Comedy,1995.0,pregnancy remake,father of the bride part ii comedy pregnancy r...


In [28]:
# Đếm số ký tự đặc biệt trong content 
special_content = movies_nlp[
    movies_nlp["content"].str.contains(
        r"[^a-zA-Z0-9\s]",
        regex=True,
        na=False
    )
]

print(len(special_content))

4858


In [29]:
# Loại bỏ ký tự đặc biệt, chỉ giữ lại chữ cái, số và khoảng trắng
movies_nlp["content"] = (
    movies_nlp["content"]
    .str.replace(
        r"[^a-zA-Z0-9\s]",
        " ",
        regex=True
    )
)
movies_nlp.head()

,movieId,title,genres,year,tag,content
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,1995.0,pixar pixar fun,toy story adventure animation children comedy ...
1,2,Jumanji,Adventure Children Fantasy,1995.0,fantasy magic board game robin williams game,jumanji adventure children fantasy fantasy mag...
2,3,Grumpier Old Men,Comedy Romance,1995.0,moldy old,grumpier old men comedy romance moldy old
3,4,Waiting to Exhale,Comedy Drama Romance,1995.0,,waiting to exhale comedy drama romance
4,5,Father of the Bride Part II,Comedy,1995.0,pregnancy remake,father of the bride part ii comedy pregnancy r...


In [30]:
# Loại bỏ khoảng trắng thừa, ở giữa nếu có nhiều khoảng trắng sẽ thành 1 khoảng trắng, ở 2 đầu nếu có sẽ bị xóa
movies_nlp["content"] = (
    movies_nlp["content"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
movies_nlp.head()

,movieId,title,genres,year,tag,content
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,1995.0,pixar pixar fun,toy story adventure animation children comedy ...
1,2,Jumanji,Adventure Children Fantasy,1995.0,fantasy magic board game robin williams game,jumanji adventure children fantasy fantasy mag...
2,3,Grumpier Old Men,Comedy Romance,1995.0,moldy old,grumpier old men comedy romance moldy old
3,4,Waiting to Exhale,Comedy Drama Romance,1995.0,,waiting to exhale comedy drama romance
4,5,Father of the Bride Part II,Comedy,1995.0,pregnancy remake,father of the bride part ii comedy pregnancy r...


In [31]:
# Tính TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    stop_words="english"
)
tfidf_matrix = tfidf.fit_transform(
    movies_nlp["content"]
)
print(tfidf_matrix.shape)

(9742, 9801)


In [32]:
feature_names = tfidf.get_feature_names_out()

movies_nlp.head()


['00' '000' '007' '01' '04' '06' '09' '10' '100' '1000' '101' '102' '10th'
 '11' '1138' '11th' '12' '120' '127' '13' '13th' '14' '1408' '1492' '15'
 '16' '1600' '17' '174' '1776' '18' '187' '19' '1900' '1900s' '1920s'
 '1933' '1935' '1941' '1950s' '1960s' '1969' '1970s' '1972' '1975' '1980s'
 '1984' '1985' '1990' '1990s' '1992' '1st' '20' '200' '2000' '2001' '2006'
 '2007' '2010' '2012' '2018' '2046' '2048' '2049' '21' '211' '22' '23'
 '24' '25' '250' '25th' '27' '28' '281' '2d' '2nd' '30' '300' '3000' '31'
 '33' '34th' '35' '37th' '39' '3d' '3dd' '40' '400' '42' '42nd' '43' '44'
 '45' '451' '46' '47' '48' '49' '4th' '50' '500' '51' '52' '54' '57' '571'
 '5th' '60' '61' '66' '6th' '70' '70mm' '71' '73' '77' '777' '7th' '80'
 '800' '81' '84' '8mm' '8th' '90' '900' '911' '93' '96' '964' '99' '9to5'
 'aan' 'aardman' 'aaron' 'abandoned' 'abbey' 'abbott' 'abbotts' 'abcs'
 'abduction' 'abiding' 'abominable' 'abortion' 'abracadabra' 'abraham'
 'abre' 'abroad' 'absence' 'absent' 'absentia' 'ab

In [33]:
## 5. Trích chọn đặc trưng TF-IDF

TF-IDF biến nội dung phim dạng chữ thành vector số.

Sau bước này:

- Mỗi dòng tương ứng với một phim.
- Mỗi cột tương ứng với một từ/đặc trưng trong từ vựng.
- Giá trị trong ma trận thể hiện mức độ quan trọng của từ đó với phim.

Tham số `stop_words="english"` giúp loại bỏ các từ tiếng Anh phổ biến như `the`, `a`, `and`, `of`,... vì các từ này ít giúp phân biệt nội dung phim.

In [34]:
# Tạo Series để map index với title, dùng drop_duplicates để tránh trường hợp có phim trùng tên
indices = pd.Series(
    movies_nlp.index,
    index=movies_nlp["title"]
).drop_duplicates()
indices.head()

title
Toy Story                      0
Jumanji                        1
Grumpier Old Men               2
Waiting to Exhale              3
Father of the Bride Part II    4
dtype: int64

In [35]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies_nlp["content"]
)

print("Kích thước ma trận TF-IDF:", tfidf_matrix.shape)

feature_names = tfidf.get_feature_names_out()

print("200 từ đầu tiên trong từ vựng:")
print(feature_names[:200])


## 6. Chia train/dev/test

Dữ liệu rating được chia thành 3 tập:

- `train`: dùng để huấn luyện SVD và xây dựng hồ sơ sở thích người dùng.
- `dev`: dùng để thử nhiều giá trị alpha và chọn cấu hình tốt.
- `test`: dùng để đánh giá cuối cùng.

Tỷ lệ chia:

```text
70% train - 15% dev - 15% test
```

In [36]:
# Test hàm đề xuất
recommend_movies("Toy Story")

,title,genres
2355,Toy Story 2,Adventure Animation Children Comedy Fantasy
1757,"Bug's Life, A",Adventure Animation Children Comedy
7355,Toy Story 3,Adventure Animation Children Comedy Fantasy IMAX
3595,"Toy, The",Comedy
3733,Fun,Crime Drama
7039,Up,Adventure Animation Children Drama
2539,We're Back! A Dinosaur's Story,Adventure Animation Children Fantasy
4089,Toy Soldiers,Action Drama
2227,"Story of Us, The",Comedy Drama
1617,"NeverEnding Story, The",Adventure Children Fantasy


In [37]:
!pip install scikit-surprise


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
# =====================================================================
# NHÁNH 2: TRÍCH CHỌN ĐẶC TRƯNG ẨN SVD (HÀNH VI NGƯỜI DÙNG)
# =====================================================================

from surprise import Dataset, Reader, SVD
from sklearn.model_selection import train_test_split


# 1. Chia dữ liệu thành 3 tập: Train / Dev / Test
# Chia lần 1: 70% train, 30% còn lại
ratings_train, ratings_temp = train_test_split(
    ratings_svd,
    test_size=0.30,
    random_state=42
)

ratings_dev, ratings_test = train_test_split(
    ratings_temp,
    test_size=0.50,
    random_state=42
)

print("--- KÍCH THƯỚC DỮ LIỆU SAU KHI CHIA ---")
print("Train:", ratings_train.shape)
print("Dev:", ratings_dev.shape)
print("Test:", ratings_test.shape)

print("Tỷ lệ Train:", len(ratings_train) / len(ratings_svd))
print("Tỷ lệ Dev:", len(ratings_dev) / len(ratings_svd))
print("Tỷ lệ Test:", len(ratings_test) / len(ratings_svd))


--- KÍCH THƯỚC DỮ LIỆU SAU KHI CHIA ---
Train: (70585, 3)
Dev: (15125, 3)
Test: (15126, 3)
--- KẾT QUẢ TRÍCH CHỌN ĐẶC TRƯNG SVD ---
Kích thước ma trận đặc trưng ẩn Người dùng (P): (610, 100)
Vector đặc trưng ẩn của User đầu tiên (10 chiều đầu): 
[ 0.01463987  0.04168454 -0.01703575  0.24802898 -0.14470641 -0.11747417
  0.15974958  0.0598824  -0.20672258  0.17191122]

Kích thước ma trận đặc trưng ẩn Bộ phim (Q): (8566, 100)
Vector đặc trưng ẩn của Movie đầu tiên (10 chiều đầu): 
[-0.06918747 -0.00297461  0.00617664 -0.01568725  0.05520322  0.00063222
  0.16806902  0.00210781  0.042894   -0.23010316]


## 7. Huấn luyện SVD

Phần này huấn luyện mô hình SVD trên tập `ratings_train`.

SVD học từ dữ liệu:

```text
userId, movieId, rating
```

Mục tiêu là dự đoán rating mà user có thể dành cho các phim chưa xem.

Các hyper-parameter đang dùng:

- `n_factors=50`: số chiều đặc trưng ẩn.
- `n_epochs=20`: số vòng học.
- `lr_all=0.005`: tốc độ học.
- `reg_all=0.02`: hệ số giảm overfitting.
- `random_state=42`: cố định kết quả khi chạy lại.

In [39]:
required_variables = [
    "ratings_svd",
    "ratings_train",
    "ratings_dev",
    "ratings_test",
    "movies_nlp",
    "tfidf_matrix",
    "svd_model"
]

for var_name in required_variables:
    if var_name not in globals():
        raise NameError(
            f"Bạn cần chạy các cell phía trên trước. Biến còn thiếu: {var_name}"
        )

print("Các biến cần thiết đã sẵn sàng.")
print("ratings_train:", ratings_train.shape)
print("ratings_dev:", ratings_dev.shape)
print("ratings_test:", ratings_test.shape)
print("movies_nlp:", movies_nlp.shape)
print("tfidf_matrix:", tfidf_matrix.shape)


Các biến cần thiết đã sẵn sàng.
ratings_svd: (100836, 3)
ratings_train: (70585, 3)
ratings_dev: (15125, 3)
ratings_test: (15126, 3)
movies_nlp: (9742, 6)
tfidf_matrix: (9742, 9801)


In [40]:
def evaluate_svd_model(model, ratings_df):
    y_true = []
    y_pred = []

    for row in ratings_df.itertuples():
        user_id = row.userId
        movie_id = row.movieId
        true_rating = row.rating

        predicted_rating = model.predict(
            user_id,
            movie_id
        ).est

        y_true.append(true_rating)
        y_pred.append(predicted_rating)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    return rmse, mae


In [41]:
dev_rmse, dev_mae = evaluate_svd_model(
    svd_model,
    ratings_dev
)

test_rmse, test_mae = evaluate_svd_model(
    svd_model,
    ratings_test
)

svd_error_results = pd.DataFrame([
    {
        "Dataset": "Dev",
        "RMSE": dev_rmse,
        "MAE": dev_mae
    },
    {
        "Dataset": "Test",
        "RMSE": test_rmse,
        "MAE": test_mae
    }
])

svd_error_results


,genre,count,avg_rating
0,Film-Noir,1,5.000000
1,Musical,20,4.850000
2,Mystery,13,4.846154
3,Animation,27,4.814815
4,Children,37,4.756757
5,Drama,64,4.656250
6,War,20,4.650000
7,Crime,39,4.615385
8,Adventure,74,4.608108
9,Thriller,43,4.581395


## 9. Xây dựng Hybrid Recommendation

Phần này xây dựng mô hình Hybrid theo đúng ý tưởng chính của đề tài:

```text
Input: userId
Output: Top N phim gợi ý cho user đó
```

Không dùng kiểu nhập tên phim rồi tìm phim tương tự, vì đó chỉ là Content-Based đơn giản.

Mô hình Hybrid gồm 2 điểm:

1. `SVDScore_norm`: điểm rating dự đoán từ SVD, chuẩn hóa về 0 đến 1.
2. `ContentScore`: điểm giống nhau giữa hồ sơ sở thích user và nội dung phim.

Công thức:

```text
HybridScore = alpha × SVDScore_norm + (1 - alpha) × ContentScore
```

### 9.1. Cấu hình chung cho Hybrid

- `RELEVANCE_THRESHOLD = 3.5`: rating từ 3.5 trở lên được xem là phim user thích.
- `K = 10`: đánh giá Top-10 phim.
- `ALPHA_LIST = [0.5, 0.7, 0.9]`: các giá trị alpha để thử trên tập dev.

In [42]:
RELEVANCE_THRESHOLD = 3.5
K = 10
ALPHA_LIST = [0.5, 0.7, 0.9]


### 9.2. Tạo các biến tra cứu

Các biến này giúp quá trình gợi ý nhanh và rõ ràng hơn:

- `user_train_items`: lưu các phim mà mỗi user đã rating trong tập train.
- `movie_id_to_index`: ánh xạ từ `movieId` sang index trong `tfidf_matrix`.

In [ ]:
user_train_items = (
    ratings_train
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

movie_id_to_index = pd.Series(
    movies_nlp.index,
    index=movies_nlp["movieId"]
).to_dict()

print("Số user trong Train:", len(user_train_items))
print("Số phim trong movie_id_to_index:", len(movie_id_to_index))


### 9.3. Chuẩn hóa điểm SVD

SVD dự đoán rating trong khoảng 0.5 đến 5.0. Nhưng `ContentScore` nằm trong khoảng 0 đến 1.  
Vì vậy cần chuẩn hóa SVD về 0 đến 1 để có thể cộng với ContentScore.

In [43]:
def normalize_svd_score(score, min_rating=0.5, max_rating=5.0):
    normalized_score = (score - min_rating) / (max_rating - min_rating)
    return max(0, min(1, normalized_score))


### 9.4. Xây dựng hồ sơ sở thích người dùng

Hệ thống lấy các phim user rating cao trong `ratings_train`, sau đó lấy vector TF-IDF của các phim đó.

Hồ sơ user được tính bằng trung bình có trọng số:

```text
UserProfile = trung bình các vector phim user thích, có trọng số là rating
```

Phim rating cao hơn sẽ ảnh hưởng mạnh hơn đến hồ sơ sở thích.

In [ ]:
def build_user_profile(
    user_id,
    ratings_df,
    threshold=3.5
):
    liked_ratings = ratings_df[
        (ratings_df["userId"] == user_id) &
        (ratings_df["rating"] >= threshold)
    ]

    if liked_ratings.empty:
        return None

    movie_vectors = []
    weights = []

    for row in liked_ratings.itertuples():
        movie_id = row.movieId

        if movie_id in movie_id_to_index:
            movie_index = movie_id_to_index[movie_id]
            movie_vectors.append(tfidf_matrix[movie_index])
            weights.append(row.rating)

    if len(movie_vectors) == 0:
        return None

    movie_matrix = vstack(movie_vectors)
    weights = np.array(weights)

    # Nhân mỗi vector phim với rating tương ứng, sau đó lấy trung bình có trọng số
    user_profile = movie_matrix.multiply(weights[:, None]).sum(axis=0) / weights.sum()

    return np.asarray(user_profile)


user_profile_demo = build_user_profile(
    user_id=1,
    ratings_df=ratings_svd,
    min_rating=3.5
)

print("Kích thước user profile:", user_profile_demo.shape)


Tính ContentScore cho từng phim
ContentScore trả lời câu hỏi:

Phim này có giống gu nội dung của user không?

Cách tính:

So sánh vector user_profile với vector TF-IDF của toàn bộ phim.
Dùng Cosine Similarity để lấy điểm từ 0 đến 1.
Điểm càng cao thì phim càng giống sở thích nội dung của user.

In [44]:
def get_user_content_scores(user_id, ratings_df, min_rating=3.5):
    user_profile = build_user_profile(
        user_id=user_id,
        ratings_df=ratings_df,
        min_rating=min_rating
    )

    return np.asarray(user_profile)


Số điểm ContentScore của user 1: 9742
5 điểm đầu: [0.19856152 0.15234492 0.10885689 0.10665674 0.05695615]


### 9.5. Hàm gợi ý phim cho một user

Hàm này là hàm chính của hệ thống.

Với một `userId`, hàm sẽ:

1. Lấy danh sách phim user đã rating trong train.
2. Loại các phim đó ra khỏi danh sách gợi ý.
3. Tính điểm SVD cho từng phim chưa xem.
4. Tính điểm nội dung nếu user có đủ phim rating cao để tạo hồ sơ.
5. Tính `hybrid_score`.
6. Sắp xếp và trả về Top N phim.

In [45]:
def recommend_movies_for_user(
    user_id,
    alpha=0.7,
    top_n=10,
    threshold=3.5,
    method="hybrid"
):
    rated_items = user_train_items.get(user_id, set())

    candidate_movies = movies_nlp[
        ~movies_nlp["movieId"].isin(rated_items)
    ].copy()

    user_profile = build_user_profile(
        user_id=user_id,
        ratings_df=ratings_train,
        threshold=threshold
    )

    if user_profile is not None:
        content_scores = cosine_similarity(
            user_profile,
            tfidf_matrix
        ).flatten()
    else:
        content_scores = None

    recommendations = []

    for row in candidate_movies.itertuples():
        movie_id = row.movieId

        svd_pred_rating = svd_model.predict(
            user_id,
            movie_id
        ).est

        svd_score_norm = normalize_svd_score(
            svd_pred_rating
        )

        if content_scores is not None and movie_id in movie_id_to_index:
            content_score = content_scores[
                movie_id_to_index[movie_id]
            ]
        else:
            content_score = 0

        hybrid_score = (
            alpha * svd_score_norm
            + (1 - alpha) * content_score
        )

        if method == "svd":
            final_score = svd_score_norm
        elif method == "hybrid":
            final_score = hybrid_score
        else:
            raise ValueError("method phải là 'svd' hoặc 'hybrid'.")

        recommendations.append({
            "movieId": movie_id,
            "title": row.title,
            "genres": row.genres,
            "svd_pred_rating": svd_pred_rating,
            "svd_score_norm": svd_score_norm,
            "content_score": content_score,
            "hybrid_score": hybrid_score,
            "final_score": final_score
        })

    recommendations_df = pd.DataFrame(recommendations)

    if recommendations_df.empty:
        return pd.DataFrame({
            "message": [f"Không còn phim phù hợp để gợi ý cho user {user_id}."]
        })

    return (
        recommendations_df
        .sort_values("final_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


## 10. Đánh giá Top-K

RMSE/MAE chỉ đánh giá SVD dự đoán rating lệch bao nhiêu.  
Nhưng hệ gợi ý còn cần đánh giá danh sách Top phim gợi ý.

Phần này dùng 3 chỉ số:

- `Precision@K`: trong K phim gợi ý, có bao nhiêu phim thật sự phù hợp.
- `Recall@K`: trong các phim phù hợp của user, hệ thống tìm lại được bao nhiêu phim.
- `NDCG@K`: đánh giá cả độ đúng và thứ tự xếp hạng. Phim phù hợp nằm càng cao thì NDCG càng tốt.

Tập `dev` dùng để chọn alpha tốt nhất.  
Tập `test` dùng để đánh giá cuối cùng.

### 10.1. Lấy phim phù hợp trong dev/test

Trong phần đánh giá Top-K, phim được coi là phù hợp nếu user rating từ `RELEVANCE_THRESHOLD` trở lên.

In [46]:
demo_user_id = 1

favorite_genres = get_user_favorite_genres(
    user_id=demo_user_id,
    ratings_df=ratings_train,
    movies_df=movies_nlp,
    min_rating=RELEVANCE_THRESHOLD
)

favorite_genres.head(10)


Các thể loại user có xu hướng thích:


,genre,count,avg_rating
0,Mystery,10,4.800000
1,Musical,14,4.785714
2,War,16,4.625000
3,Animation,13,4.615385
4,Children,23,4.608696
5,Drama,49,4.591837
6,Crime,30,4.566667
7,Adventure,53,4.547170
8,Action,55,4.527273
9,Comedy,57,4.526316


Top phim gợi ý theo Hybrid:


,movieId,title,genres,svd_pred_rating,svd_score_norm,content_score,hybrid_score
0,48774,Children of Men,Action Adventure Drama Sci-Fi Thriller,4.788953,0.953101,0.554966,0.833660
1,78499,Toy Story 3,Adventure Animation Children Comedy Fantasy IMAX,4.906543,0.979232,0.298108,0.774895
2,55247,Into the Wild,Action Adventure Drama,4.773284,0.949619,0.322283,0.761418
3,1199,Brazil,Fantasy Sci-Fi,5.000000,1.000000,0.204420,0.761326
4,59315,Iron Man,Action Adventure Sci-Fi,4.654515,0.923225,0.364647,0.755652
5,1223,"Grand Day Out with Wallace and Gromit, A",Adventure Animation Children Comedy Sci-Fi,4.869351,0.970967,0.238932,0.751356
6,60684,Watchmen,Action Drama Mystery Sci-Fi Thriller IMAX,4.743262,0.942947,0.286338,0.745964
7,159093,Now You See Me 2,Action Comedy Thriller,4.357193,0.857154,0.486440,0.745940
8,7143,"Last Samurai, The",Action Adventure Drama War,4.653129,0.922918,0.329399,0.744862
9,87232,X-Men: First Class,Action Adventure Sci-Fi Thriller War,4.478796,0.884177,0.409501,0.741774


In [47]:
demo_recommendations = recommend_movies_for_user(
    user_id=demo_user_id,
    alpha=best_alpha,
    top_n=10,
    threshold=RELEVANCE_THRESHOLD,
    method="hybrid"
)

demo_recommendations[
    [
        "movieId",
        "title",
        "genres",
        "svd_pred_rating",
        "svd_score_norm",
        "content_score",
        "hybrid_score"
    ]
]


Top 5 phim gợi ý cho user 1 với alpha = 0.5


,title,genres,svd_pred_rating,svd_score_norm,content_score,hybrid_score
0,Children of Men,Action Adventure Drama Sci-Fi Thriller,4.788953,0.953101,0.554966,0.754034
1,Eight Below,Action Adventure Drama Romance,4.119496,0.804332,0.555026,0.679679
2,Now You See Me 2,Action Comedy Thriller,4.357193,0.857154,0.486440,0.671797
3,D.A.R.Y.L.,Adventure Children Sci-Fi,4.162812,0.813958,0.486672,0.650315
4,To Have and Have Not,Adventure Drama Romance Thriller War,4.194333,0.820963,0.476305,0.648634


Top 5 phim gợi ý cho user 1 với alpha = 0.7


,title,genres,svd_pred_rating,svd_score_norm,content_score,hybrid_score
0,Children of Men,Action Adventure Drama Sci-Fi Thriller,4.788953,0.953101,0.554966,0.833660
1,Toy Story 3,Adventure Animation Children Comedy Fantasy IMAX,4.906543,0.979232,0.298108,0.774895
2,Into the Wild,Action Adventure Drama,4.773284,0.949619,0.322283,0.761418
3,Brazil,Fantasy Sci-Fi,5.000000,1.000000,0.204420,0.761326
4,Iron Man,Action Adventure Sci-Fi,4.654515,0.923225,0.364647,0.755652


Top 5 phim gợi ý cho user 1 với alpha = 0.9


,title,genres,svd_pred_rating,svd_score_norm,content_score,hybrid_score
0,Brazil,Fantasy Sci-Fi,5.000000,1.000000,0.204420,0.920442
1,Children of Men,Action Adventure Drama Sci-Fi Thriller,4.788953,0.953101,0.554966,0.913287
2,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy War,5.000000,1.000000,0.118509,0.911851
3,Toy Story 3,Adventure Animation Children Comedy Fantasy IMAX,4.906543,0.979232,0.298108,0.911120
4,Kiss Kiss Bang Bang,Comedy Crime Mystery Thriller,5.000000,1.000000,0.098634,0.909863


### 11.3. So sánh gợi ý với nhiều alpha

Phần này giúp quan sát khi tăng/giảm alpha thì danh sách gợi ý thay đổi như thế nào.

In [ ]:
for alpha in ALPHA_LIST:
    print("=" * 80)
    print(f"Top 5 phim gợi ý cho user {demo_user_id} với alpha = {alpha}")

    result = recommend_movies_for_user(
        user_id=demo_user_id,
        alpha=alpha,
        top_n=5,
        threshold=RELEVANCE_THRESHOLD,
        method="hybrid"
    )

    display(
        result[
            [
                "title",
                "genres",
                "svd_pred_rating",
                "svd_score_norm",
                "content_score",
                "hybrid_score"
            ]
        ]
    )


## 12. Lưu kết quả và model

Phần này lưu:

1. Kết quả đánh giá SVD bằng RMSE/MAE.
2. Kết quả đánh giá Top-K trên dev.
3. Kết quả đánh giá Top-K trên test.
4. Dữ liệu phim đã xử lý.
5. Các file model `.pkl` để dùng cho Streamlit.

Các file `.pkl` dùng để app Streamlit load lại model mà không cần train lại từ đầu.

In [48]:
import os
import pickle

os.makedirs("movie_rcm_demo/model", exist_ok=True)

with open("movie_rcm_demo/model/svd_model.pkl", "wb") as f:
    pickle.dump(svd_model, f)

with open("movie_rcm_demo/model/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open("movie_rcm_demo/model/movies_nlp.pkl", "wb") as f:
    pickle.dump(movies_nlp, f)

with open("movie_rcm_demo/model/ratings_train.pkl", "wb") as f:
    pickle.dump(ratings_train, f)

with open("movie_rcm_demo/model/movie_id_to_index.pkl", "wb") as f:
    pickle.dump(movie_id_to_index, f)

print("Đã lưu model vào movie_rcm_demo/model")

Đã lưu model vào movie_rcm_demo/model
